# GenAI Research Paper Assistant: RAG Pipeline Starter

## 1. Introduction
Welcome to the early-stage implementation notebook for the **GenAI Research Paper Assistant**. 

This project leverages **Retrieval-Augmented Generation (RAG)** to provide accurate, context-aware answers to user queries based entirely on the contents of uploaded academic research papers. This method minimizes LLM "hallucinations" by grounding the generative model in verified literature.

In this initial notebook, we will set up the foundational steps of the RAG pipeline:
1. Installing required dependencies.
2. Loading a sample academic PDF (Lewis et al., 2020).
3. Extracting the raw text from the document.
4. Splitting the text into manageable chunks for future vector embedding.

## 2. Installation of Required Libraries

Before we begin, we need to install the necessary libraries for PDF parsing and text chunking. 
We will be using `langchain` for orchestration and `PyMuPDF` (`fitz`) for robust PDF text extraction.

In [ ]:
# Install core libraries for Document Ingestion and Chunking
# !pip install -qU langchain langchain-community PyMuPDF tiktoken

In [ ]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

## 3. Loading a PDF Research Paper

We will use the foundational paper on RAG: *"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"* by Lewis et al. 
Ensure that the paper `RAG_Lewis_2020.pdf` is located in the `../research-papers/` directory.

In [ ]:
# Define the path to the research paper PDF
pdf_path = "../research-papers/RAG_Lewis_2020.pdf"

# Check if the file exists
if os.path.exists(pdf_path):
    print(f"File found at: {pdf_path}")
else:
    print(f"Error: File not found at {pdf_path}. Please check the path.")

## 4. Extracting Text

Using Langchain's `PyMuPDFLoader`, we can easily ingest the PDF. This loader reads the document and returns a list of `Document` objects, where each object typically represents a single page of the PDF containing both the text content and useful metadata (like the page number).

In [ ]:
def load_pdf_document(file_path):
    """
    Loads a PDF document and returns a list of Langchain Document objects.
    """
    loader = PyMuPDFLoader(file_path)
    documents = loader.load()
    return documents

# Uncomment the following lines to run the extraction
# pages = load_pdf_document(pdf_path)
# print(f"Successfully loaded {len(pages)} pages from the PDF.")
# print("\n--- Sample text from Page 1 ---\n")
# print(pages[0].page_content[:500] + "...")

## 5. Splitting Text into Chunks

Language Models have a strictly limited context window. They cannot ingest a 20-page document all at once. Furthermore, our Embedding Model works best on focused paragraphs rather than entire pages.

To solve this, we split the extracted text into smaller sequences called **Chunks**. We use a `chunk_overlap` to ensure that sentences aren't cleanly cut in half across chunks, preserving contextual meaning.

In [ ]:
def chunk_documents(docs):
    """
    Splits a list of Documents into smaller overlapping chunks.
    """
    # RecursiveCharacterTextSplitter is recommended for standard text.
    # It tries to split on paragraphs, then sentences, then words if necessary.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,    # Target size of the chunk (in characters)
        chunk_overlap=200,  # Overlap to maintain context between chunks
        length_function=len # Function used to measure chunk length
    )
    
    chunks = text_splitter.split_documents(docs)
    return chunks

# Uncomment the following lines to execute chunking
# document_chunks = chunk_documents(pages)
# print(f"Split the document into {len(document_chunks)} chunks.")
# print("\n--- Sample Chunk ---\n")
# print(document_chunks[0].page_content)

---
## Ongoing Tasks for Future Implementation

The next steps for building this RAG pipeline will involve:

1. **Embedding Generation:** We need to initialize an embedding model (e.g., `sentence-transformers/all-MiniLM-L6-v2` via HuggingFace).
2. **Vector Database Setup:** We will pass our `document_chunks` through the embedding model and store them locally using a `FAISS` or `ChromaDB` index.
3. **Retriever Construction:** We will wrap our Vector Database in a retriever interface to search for the $k$-most similar chunks based on a user's prompt.
4. **LLM Integration:** Connect an LLM to generate answers grounded *only* in the chunks returned by the retriever.

*To be continued in the next development phase...*